# PySpark Performance Optimization

## Objective

### The objective of this notebook is to demonstrate how different Spark optimization techniques improve the performance of ETL workloads.

### Dataset:
- Customers: 100000 rows
- Products: 5000 rows
- Orders: 1000000 rows

In [218]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
import time



## Load the Dataset

In [219]:
spark=SparkSession.builder.appName('project').getOrCreate()
customers_df=spark.read.parquet('project_data/customers/')
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)



In [220]:
products_df=spark.read.parquet('project_data/products/')
products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)



In [221]:
orders_df=spark.read.parquet('project_data/orders/')
orders_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- price: integer (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



## Dataset Overview

The project uses a synthetic e-commerce dataset consisting of three tables

- Customers - Dimension table with 100000 rows
- Products - Dimension table with 5000 rows
- Orders - Fact table with 1000000 rows , partitioned by year and month.

## Data Model

Customers (1)-----------< Orders >-----------(1) Products

- customer_id -> Foreign key in Orders
- product_id -> Foreign key in Orders

## Optimization Techniques

### 1) Broadcast Join Optimization

#### Objective

Demonstrate how broadcasting a small dimension table eliminates unnecessary shuffling and improves join performance

#### Business Scenario

**Calculate the total sales by product category.**

- Orders contains 1000000 records
- Products contains 5000 records

Since the Products table is smaller than the Orders table, it is a good candidate for broadcast join

###  1.1 Baseline Implementation

Note: Spark automatically broadcasts small table when spark.sql.autoBroadcastJoinThreshold permits. To clearly demonstrate the performance difference , a merge join hint was used in the baseline implementation to force a Sort Merge Join

In [222]:
baseline_df=orders_df.hint("merge") \
    .join(products_df.hint("merge"), "product_id") \
    .groupBy("category") \
    .agg(sum("total_amount").alias("total_sales"))



In [223]:
start=time.time()
baseline_df.show()
end=time.time()
print(f'Execution Time {end-start:.2f} seconds')


+---------------+-----------+
|       category|total_sales|
+---------------+-----------+
|          books|  308944879|
|home_appliances|  293295428|
|    electronics|  315029652|
|         sports|  287060679|
|       clothing|  292898951|
+---------------+-----------+

Execution Time 3.28 seconds


In [224]:
baseline_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'sum('total_amount) AS total_sales#11793]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, product_name#11779, category#11780, price#11781]
   +- Join Inner, (product_id#11782 = product_id#11778)
      :- ResolvedHint (strategy=merge)
      :  +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet
      +- ResolvedHint (strategy=merge)
         +- Relation [product_id#11778,product_name#11779,category#11780,price#11781] parquet

== Analyzed Logical Plan ==
category: string, total_sales: bigint
Aggregate [category#11780], [category#11780, sum(total_amount#11788) AS total_sales#11793L]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#1178

### Observation

#### SortMergeJoin requires both sides to be shuffled by the join key and then sorted this means [N] bytes transferred across the network. In the physical plan you can see two Exchange hashpartitioning nodes, one for each table. BroadcastHashJoin eliminates both shuffles by sending the 5K-row products table to every executor in memory.

### 1.2 Optimized Implementation

#### Broadcast the Products table  , since it is significantly smaller than the Orders table , spark can send Products table to every executor. This eliminates the shuffle required for a Sort Merge Join.

In [225]:
optimized_df=orders_df.join(broadcast(products_df), "product_id") \
    .groupBy("category") \
    .agg(sum("total_amount").alias("total_sales"))


In [226]:
start=time.time()
optimized_df.show()
end=time.time()
print(f'Execution Time {end-start:.2f} seconds')
optimized_df.explain(True)


+---------------+-----------+
|       category|total_sales|
+---------------+-----------+
|          books|  308944879|
|    electronics|  315029652|
|home_appliances|  293295428|
|         sports|  287060679|
|       clothing|  292898951|
+---------------+-----------+

Execution Time 1.85 seconds
== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'sum('total_amount) AS total_sales#11819]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, product_name#11779, category#11780, price#11781]
   +- Join Inner, (product_id#11782 = product_id#11778)
      :- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet
      +- ResolvedHint (strategy=broadcast)
         +- Relation [product_id#11778,product_name#11779,category#11780,price#11781] parquet

== Anal

### Observation

#### The physical plan uses BroadcastHashJoin , eliminates shuffle for the large Orders table, reduces network communication, improves query execution time.

### 2) Predicate Pushdown

### Objective

- #### Demonstrate how spark pushes filter conditions down to the Parquet Reader to minimize disk I/O.
- #### When a filter can be pushed down, Spark reads only the required data instead of Scanning unnecessary rows.
- #### In this example , we compare a filter that prevents predicate pushdown with one that allows it.

### 2.1) Baseline Implementation

### In this implementation , the abs() function is applied to the customer_id column before filtering .Since the filter expression contains a function, Spark cannot push the filter directly to the Parquet data source. As a result, Spark reads the data first and applies the filter afterward.

In [227]:
baseline_df = orders_df.filter(abs(col("customer_id")) == 100)

In [228]:
start=time.time()
baseline_df.count()
end=time.time()
print(f'Execution Time {end-start:.2f} seconds')
baseline_df.explain(True)

Execution Time 0.55 seconds
== Parsed Logical Plan ==
'Filter '`=`('abs('customer_id), 100)
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
product_id: int, order_id: bigint, customer_id: int, quantity: int, order_date: date, price: int, total_amount: int, order_year: int, order_month: int
Filter (abs(customer_id#11784) = 100)
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Filter (isnotnull(customer_id#11784) AND (abs(customer_id#11784) = 100))
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Physical Plan ==
*(1) Filter (isnotnull(customer_id#11784) AND (abs(cu

### 2.2) Optimized Implementation

### Instead of applying a function to the column, filter directly on the original column value. This allows Spark to push the filter down to the parquet reader, reducing the amount of data needs to be processed

In [229]:
optimized_df=orders_df.filter(col("customer_id") == 100)

In [230]:
start=time.time()
optimized_df.count()
end=time.time()
print(f'Execution Time {end-start:.2f} seconds')
optimized_df.explain(True)


Execution Time 1.02 seconds
== Parsed Logical Plan ==
'Filter '`=`('customer_id, 100)
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
product_id: int, order_id: bigint, customer_id: int, quantity: int, order_date: date, price: int, total_amount: int, order_year: int, order_month: int
Filter (customer_id#11784 = 100)
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Filter (isnotnull(customer_id#11784) AND (customer_id#11784 = 100))
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Physical Plan ==
*(1) Filter (isnotnull(customer_id#11784) AND (customer_id#11784 = 10

### Observation

#### In the optimized implementation notice the DictionaryFilters DictionaryFilters: [(customer_id#11330 = 100)] whereas in baseline implementation it is DictionaryFilters: []. Parquet stores columns using dictionary encoding. When the filter is a direct equality on the raw column value, the Parquet reader can use the dictionary to skip entire row groups without deserialising them. abs(customer_id) breaks this because the transformation must be applied to each value before comparison the reader can't use the dictionary.

### 3) Catalyst Optimizer Automatic Column Pruning

Objective
- Demonstrate how Spark reads only required columns from Parquet files.
- Reduce I/O by avoiding unnecessary columns.
- Show how Catalyst Optimizer automatically performs column pruning.

In [231]:
column_df=orders_df.join(products_df,'product_id') \
    .groupBy('category') \
    .agg(sum('total_amount').alias('total_sales'))

column_df.explain(True)



== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'sum('total_amount) AS total_sales#11871]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, product_name#11779, category#11780, price#11781]
   +- Join Inner, (product_id#11782 = product_id#11778)
      :- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet
      +- Relation [product_id#11778,product_name#11779,category#11780,price#11781] parquet

== Analyzed Logical Plan ==
category: string, total_sales: bigint
Aggregate [category#11780], [category#11780, sum(total_amount#11788) AS total_sales#11871L]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, product_name#11779, cate

### Observation

The physical plan shows:
- **PhotonProject [product_id#11195, total_amount#11201**

which means Spark keeps only the columns required for the join and aggregation. Although the source table contains additional columns, the scan ultimately reads only product_id and total_amount (ReadSchema: struct<product_id:int,total_amount:int>), demonstrating Column Pruning and reducing unnecessary I/O.



### 4) Partition Pruning


#### Objective

Demonstrate how filtering on partition column allows spark to scan only the required partitions instead of the entire dataset.

The orders dataset is partitioned by order_year and order_month. Using these columns in filter conditions reduces the amount of data read from storage.


### 4.1) Baseline Implementation


##### This implementation filters using the order_date column. Although the result is correct, Spark cannot use the partition directory structure as effectively because the filter is not directly on the partition columns.

In [232]:
baseline_df=orders_df.filter(year(col('order_date'))==2025) \
    .groupBy('product_id') \
    .agg(sum(col('total_amount')).alias('total_sales'))

In [233]:
start=time.time()
baseline_df.count()
end=time.time()
baseline_time=end-start
print(f"Execution time {end-start:.2f} seconds")

Execution time 1.24 seconds


In [234]:
baseline_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('total_amount) AS total_sales#11887]
+- Filter (year(order_date#11786) = 2025)
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
product_id: int, total_sales: bigint
Aggregate [product_id#11782], [product_id#11782, sum(total_amount#11788) AS total_sales#11887L]
+- Filter (year(order_date#11786) = 2025)
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Aggregate [product_id#11782], [product_id#11782, sum(total_amount#11788) AS total_sales#11887L]
+- Project [product_id#11782, total_amount#11788]
   +- Filter (isnotnull(order_date#11786) AND (year(order_date#11786) = 2025))
      +- Relation [product_id#11782,order_i


### Observation

Inspect the physical execution plan.

Check the **PartitionFilters[]** section.

Since the filter is applied using year(order_date) , Spark cannot directly use the partition columns for pruning.


### 4.2) Optimized Implementation

Filter directly on the partition column order_year.

Since the dataset is partitioned by order_year , Spark reads only the matching partition instead of scanning all partitions

In [235]:
optimized_df = orders_df \
  .filter(col('order_year')==2025) \
  .groupBy('product_id') \
  .agg(sum(col('total_amount'))
    .alias('total_sales'))


In [236]:
start=time.time()
optimized_df.count()
end=time.time()
print(f"Execution time {end-start:.2f} seconds")


Execution time 0.69 seconds


In [237]:
optimized_df.explain(True)


== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('total_amount) AS total_sales#11906]
+- Filter (order_year#11789 = 2025)
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
product_id: int, total_sales: bigint
Aggregate [product_id#11782], [product_id#11782, sum(total_amount#11788) AS total_sales#11906L]
+- Filter (order_year#11789 = 2025)
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Aggregate [product_id#11782], [product_id#11782, sum(total_amount#11788) AS total_sales#11906L]
+- Project [product_id#11782, total_amount#11788]
   +- Filter (isnotnull(order_year#11789) AND (order_year#11789 = 2025))
      +- Relation [product_id#11782,order_id#11783L,customer_


### Observation

Inspect the **PartitionFilters** section in the physical plan.
- **PartitionFilters: [isnotnull(order_year#11202), (order_year#11202 = 2025)]**

This indicates that spark is reading only the partitions for the year 2025 instead of scanning the entire dataset.The orders dataset is stored as order_year=2023/, order_year=2024/, order_year=2025/ directories. Filtering on col('order_year')==2025 tells Spark to open only order_year=2025/ it never reads from 2023 or 2024 directories. This is a file system level skip, before any data is deserialised. Filtering on year(order_date) bypasses this because the partition key is order_year, not order_date Spark must open all directories and evaluate year(order_date) on each row

### 5) Data Skew

#### Objective

Demonstrate how uneven data distribution across partitions causes performance bottlenecks,
and how the salting technique resolves it by breaking hot keys into smaller chunks.

#### What is Data Skew?

In a distributed system, Spark splits data across partitions and processes them in parallel.
Data skew occurs when one partition receives significantly more data than others usually
because a small number of keys appear disproportionately often in the dataset.

The result: one executor processes millions of rows while others sit idle. The entire
stage waits for the slowest task to finish. No amount of adding more executors fixes this the bottleneck is within a single partition.

#### Business Scenario

Simulate a skewed orders dataset where a small number of customers account for 60% of all
orders (e.g. bulk buyers or bot accounts). Calculate total spend per customer and observe
the impact on task distribution.

### 5.1) Inject Skew into the Dataset

In [238]:
# Artificially skew the data so that customer_id = 1, 2, 3
# account for 60% of all 1M orders
# This simulates a real-world hot key scenario (bulk buyers, bot traffic)

skewed_orders = orders_df.withColumn(
    "customer_id",
    when(rand() < 0.6,
         (rand() * 3 + 1).cast("int"))   # 60% of rows → customer_id 1, 2 or 3
    .otherwise(col("customer_id"))        # remaining 40% keep original customer_id
)

# Verify the skew — top customers by order count
skewed_orders.groupBy("customer_id") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(5)

+-----------+------+
|customer_id| count|
+-----------+------+
|          3|200049|
|          1|199861|
|          2|199812|
|      47379|    14|
|      93152|    14|
+-----------+------+
only showing top 5 rows


### 5.2) Baseline — GroupBy on Skewed Data

Without any fix, all rows for customer_id = 1, 2, 3 land on the same partition.
One executor processes ~600,000 rows while others process a few hundred.

In [239]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
# Disable AQE so skew is visible — AQE has a skew join handler
# that would silently fix it and hide the problem

t0 = time.time()

skewed_result = skewed_orders \
    .groupBy("customer_id") \
    .agg(sum("total_amount").alias("total_spend")) \
    .orderBy(col("total_spend").desc())

skewed_result.show(5)
print(f"Baseline (skewed) time: {time.time()-t0:.2f}s")

skewed_result.explain(True)
# Open Spark UI → Stages → look at task duration bar
# One task will run significantly longer than all others

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          2|  299554708|
|          1|  299363752|
|          3|  299354250|
|       5563|      40488|
|      46976|      39556|
+-----------+-----------+
only showing top 5 rows
Baseline (skewed) time: 1.71s
== Parsed Logical Plan ==
'Sort ['total_spend DESC NULLS LAST], true
+- Aggregate [customer_id#11925], [customer_id#11925, sum(total_amount#11788) AS total_spend#11946L]
   +- Project [product_id#11782, order_id#11783L, CASE WHEN (rand(5271048323075733370) < 0.6) THEN cast(((rand(-2771006047555893570) * cast(3 as double)) + cast(1 as double)) as int) ELSE customer_id#11784 END AS customer_id#11925, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790]
      +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==


### 5.3) Optimized — Salting Technique

Salting breaks a hot key into multiple sub-keys by appending a random number (the "salt").
This spreads rows that originally mapped to one partition across multiple partitions,
eliminating the bottleneck.

The aggregation is done in two steps:
1. Group by the salted key → partial aggregation (spread across many partitions)
2. Strip the salt → group by original key → final aggregation (small, fast)

In [240]:
SALT_BUCKETS = 10

t0 = time.time()  # ← start timer HERE, before everything

# Step 1: add salt
salted_orders = skewed_orders.withColumn(
    "salted_key",
    concat(
        col("customer_id").cast("string"),
        lit("_"),
        (rand() * SALT_BUCKETS).cast("int").cast("string")
    )
)

# Step 2: partial aggregation on salted key
partial_agg = salted_orders \
    .groupBy("salted_key") \
    .agg(sum("total_amount").alias("partial_spend"))

# Step 3: strip salt → final aggregation
final_result = partial_agg \
    .withColumn("customer_id",
        split(col("salted_key"), "_")[0].cast("int")) \
    .groupBy("customer_id") \
    .agg(sum("partial_spend").alias("total_spend")) \
    .orderBy(col("total_spend").desc())

final_result.show(5)
print(f"Optimized (salted) time: {time.time()-t0:.2f}s")

final_result.explain(True)

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|          2|  299554708|
|          1|  299363752|
|          3|  299354250|
|       5563|      40488|
|      46976|      39556|
+-----------+-----------+
only showing top 5 rows
Optimized (salted) time: 5.24s
== Parsed Logical Plan ==
'Sort ['total_spend DESC NULLS LAST], true
+- Aggregate [customer_id#11983], [customer_id#11983, sum(partial_spend#11971L) AS total_spend#11984L]
   +- Project [salted_key#11970, partial_spend#11971L, cast(split(salted_key#11970, _, -1)[0] as int) AS customer_id#11983]
      +- Aggregate [salted_key#11970], [salted_key#11970, sum(total_amount#11788) AS partial_spend#11971L]
         +- Project [product_id#11782, order_id#11783L, customer_id#11925, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, concat(cast(customer_id#11925 as string), _, cast(cast((rand(8996287738664445978) * cast(10 as double)) as int) as string)) AS sa

### 5.4) Alternative — AQE Skew Join Handler (Spark 3.x)

Spark 3.x introduced Adaptive Query Execution (AQE) which can detect and handle skewed
partitions automatically during joins without any code changes.


In [241]:
# Re-enable AQE and turn on skew join handling
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64mb")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
# A partition is considered skewed if it is 5x larger than the median partition size

t0 = time.time()

aqe_result = skewed_orders \
    .join(customers_df, "customer_id") \
    .groupBy("state") \
    .agg(sum("total_amount").alias("total_spend"))

aqe_result.show(5)
print(f"AQE time: {time.time()-t0:.2f}s")

aqe_result.explain(True)
# Look for: CustomShuffleReader with coalesced/split notation
# AQE splits the large skewed partition into smaller sub-partitions at runtime

+--------------+-----------+
|         state|total_spend|
+--------------+-----------+
|          Utah|   12158719|
|          Ohio|   12215066|
|       Indiana|   11690597|
|North Carolina|   12290287|
|       Florida|   12375219|
+--------------+-----------+
only showing top 5 rows
AQE time: 2.72s
== Parsed Logical Plan ==
'Aggregate ['state], ['state, 'sum('total_amount) AS total_spend#12011]
+- Project [customer_id#11925, product_id#11782, order_id#11783L, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, customer_name#11774, city#11775, state#11776, country#11777]
   +- Join Inner, (customer_id#11925 = customer_id#11773)
      :- Project [product_id#11782, order_id#11783L, CASE WHEN (rand(5271048323075733370) < 0.6) THEN cast(((rand(-2771006047555893570) * cast(3 as double)) + cast(1 as double)) as int) ELSE customer_id#11784 END AS customer_id#11925, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year

### Observation

#### Why Skew is a Problem

`groupBy("customer_id").count()` confirms that customer_id 1, 2, and 3 each hold
~200,000 rows while every other customer has fewer than 20. During
`Exchange hashpartitioning(customer_id)`, all rows with the same key route to the
same partition — so 3 partitions absorb 60% of 1M rows while the rest get almost
nothing. On a cluster, those 3 executors are overwhelmed while all others sit idle.
The stage cannot finish until the slowest partition completes. Adding more executors
does not help — the bottleneck is inside a single partition.

#### How Salting Fixes It

Appending a random number (0–9) breaks each hot key into 10 sub-keys:
`customer_id=1` → `1_0, 1_1 ... 1_9` — each routed to a different partition.
200,000 rows that previously landed on one partition now spread across 10.
A two-step aggregation preserves correctness: partial sum on salted key,
then final sum after stripping the salt.

#### Why Timing Looks Worse on Colab

Colab is single-node — all partitions run sequentially on one executor regardless
of distribution. Salting adds two shuffles instead of one, so it appears slower
locally. On a multi-node cluster the rebalancing pays off — 10 executors working
in parallel vs 1 doing all the work. The proof here is the key distribution
output and the explain() plan, not the wall-clock time.

#### Salting vs AQE

AQE's skew handler splits oversized partitions at runtime — but only during
**joins**, not `groupBy`. If the bottleneck is a skewed aggregation, salting
is the only solution regardless of Spark version.

| | Salting | AQE Skew Join |
|---|---|---|
| groupBy skew | Yes | No |
| Join skew | Yes | Yes |
| Code change needed | Yes | No |
| Spark version | Any | 3.x+ |

### 6) Repartition vs Coalesce


### Objective

Demonstrate the difference between repartition and coalesce and understand when each should be used.

Spark stores data in partitions. The number of partitions affects task parallelism and execution performance.

This section compares repartitioning and coalescing th Orders dataset.


### 6.1) Baseline Implementation


#### Use the dataset with its default partitioning and perform an aggregation.


In [242]:
print("Default Partitions :", orders_df.rdd.getNumPartitions())

Default Partitions : 9


In [243]:
baseline_df = orders_df \
    .groupBy("order_year") \
    .agg(sum("total_amount").alias("total_sales"))

In [244]:
import time

start = time.time()

baseline_df.show()

end = time.time()

print(f"Execution Time : {end-start:.2f} seconds")

+----------+-----------+
|order_year|total_sales|
+----------+-----------+
|      2025|  499045433|
|      2026|   42307113|
|      2024|  501163315|
|      2023|  454713728|
+----------+-----------+

Execution Time : 0.58 seconds


In [245]:
baseline_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['order_year], ['order_year, 'sum('total_amount) AS total_sales#12038]
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
order_year: int, total_sales: bigint
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12038L]
+- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12038L]
+- Project [total_amount#11788, order_year#11789]
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Physical Plan ==
AdaptiveSparkPlan


### The dataset uses Spark's default partitioning. No explicit repartitioning has been applied.

## 6.2) Repartition Implementation

Increase the number of partitions using `repartition()`.

Unlike `coalesce()`, `repartition()` performs a full shuffle to evenly redistribute data across the specified number of partitions.



In [246]:
repartition_df = orders_df.repartition(20)

In [247]:
print("Partitions after repartition :", repartition_df.rdd.getNumPartitions())

Partitions after repartition : 20


In [248]:
repartition_result = (
    repartition_df
    .groupBy("order_year")
    .agg(sum("total_amount").alias("total_sales"))
)

In [249]:
start = time.time()

repartition_result.show()

end = time.time()

print(f"Execution Time : {end-start:.2f} seconds")

+----------+-----------+
|order_year|total_sales|
+----------+-----------+
|      2025|  499045433|
|      2026|   42307113|
|      2024|  501163315|
|      2023|  454713728|
+----------+-----------+

Execution Time : 2.18 seconds


In [250]:
repartition_result.explain(True)

== Parsed Logical Plan ==
'Aggregate ['order_year], ['order_year, 'sum('total_amount) AS total_sales#12060]
+- Repartition 20, true
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
order_year: int, total_sales: bigint
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12060L]
+- Repartition 20, true
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12060L]
+- Repartition 20, true
   +- Project [total_amount#11788, order_year#11789]
      +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#1178

### Observation

The physical execution plan contains two Exchange operators.

The first Exchange (`RoundRobinPartitioning`) is introduced by the explicit `repartition(20)` call, which redistributes the data evenly across 20 partitions.

The second Exchange (`hashpartitioning(order_year)`) is required by the `groupBy` operation so that all rows with the same `order_year` are brought together for aggregation.

Since `repartition()` performs a full shuffle, it introduces additional overhead before the aggregation.



The baseline implementation was faster because it used the existing 9 partitions.

Applying `repartition(20)` introduced an additional shuffle (`Exchange RoundRobinPartitioning`), redistributing the entire dataset before the aggregation.

Although repartition increased the number of partitions, the cost of the extra shuffle outweighed any benefit for this workload. This demonstrates that repartition should be used only when there is a specific requirement for redistributing data.Use repartition() when:

(1) you need to increase partitions  coalesce cannot do this.

(2) You need even distribution before a heavy join on a skewed
dataset.

(3) You want to repartition by a specific column like repartition(10, col("customer_id")) to co-locate data for downstream joins.

 Use coalesce() when: reducing partitions before writing output files to avoid many small files.

## 6.3) Coalesce Implementation

Reduce the number of partitions using `coalesce()`.

Unlike `repartition()`, `coalesce()` attempts to merge existing partitions with minimal data movement, making it more efficient when decreasing the number of partitions.

In [251]:
coalesce_df = orders_df.coalesce(2)

In [252]:
print("Partitions after coalesce :", coalesce_df.rdd.getNumPartitions())

Partitions after coalesce : 2


In [253]:
coalesce_result = (
    coalesce_df
    .groupBy("order_year")
    .agg(sum("total_amount").alias("total_sales"))
)

In [254]:
start = time.time()

coalesce_result.show()

end = time.time()

print(f"Execution Time : {end-start:.2f} seconds")

+----------+-----------+
|order_year|total_sales|
+----------+-----------+
|      2025|  499045433|
|      2026|   42307113|
|      2024|  501163315|
|      2023|  454713728|
+----------+-----------+

Execution Time : 0.57 seconds


In [255]:
coalesce_result.explain(True)

== Parsed Logical Plan ==
'Aggregate ['order_year], ['order_year, 'sum('total_amount) AS total_sales#12083]
+- Repartition 2, false
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
order_year: int, total_sales: bigint
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12083L]
+- Repartition 2, false
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Optimized Logical Plan ==
Aggregate [order_year#11789], [order_year#11789, sum(total_amount#11788) AS total_sales#12083L]
+- Repartition 2, false
   +- Project [total_amount#11788, order_year#11789]
      +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#1178

Because coalesce() merges existing partitions by combining adjacent partition files without shuffling, the resulting partitions may be unequal in size some executors may receive more data than others. This is acceptable when writing output files (where even distribution matters less), but repartition() should be preferred when even data distribution is required for downstream operations like joins.

### 6.4) Repartition by Column
 Scenario: before joining orders with customers on customer_id,
 repartition both tables on customer_id so matching keys
 land on the same partition reduces shuffle during join

### 6.4.1) Baseline

Without implementing repartition.

In [256]:
t0 = time.time()
baseline = (
    orders_df
    .join(customers_df, "customer_id")
    .groupBy("state")
    .agg(sum("total_amount").alias("total_sales"))
)
baseline.show()
print(f"Time: {time.time()-t0:.2f}s")

baseline.explain(True)

+--------------+-----------+
|         state|total_sales|
+--------------+-----------+
|          Utah|   30374790|
|          Ohio|   30525196|
|       Florida|   31004247|
|       Indiana|   29520128|
|North Carolina|   31034602|
|       Vermont|   28716560|
|       Montana|   28757725|
|      Illinois|   30059586|
|      Nebraska|   28074526|
|        Hawaii|   29991543|
|     Louisiana|   30751912|
|      Virginia|   30220875|
|      Oklahoma|   28843743|
|       Wyoming|   30308825|
|      Michigan|   29155589|
|  North Dakota|   30107726|
|        Alaska|   28323539|
|  South Dakota|   30040657|
| Massachusetts|   30024449|
|      Arkansas|   29588266|
+--------------+-----------+
only showing top 20 rows
Time: 3.71s
== Parsed Logical Plan ==
'Aggregate ['state], ['state, 'sum('total_amount) AS total_sales#12105]
+- Project [customer_id#11784, product_id#11782, order_id#11783L, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790,

### 6.4.2) Implementing Repartition

In [257]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", 10)

In [258]:
orders_repart = orders_df.repartition(10, col("customer_id"))
customers_repart = customers_df.repartition(10, col("customer_id"))

In [259]:
t0 = time.time()
result = orders_repart.join(customers_repart, "customer_id") \
    .groupBy("state") \
    .agg(sum("total_amount").alias("total_sales"))
result.show()
print(f"Time: {time.time()-t0:.2f}s")

result.explain(True)

+--------------+-----------+
|         state|total_sales|
+--------------+-----------+
|          Ohio|   30525196|
|       Florida|   31004247|
|       Indiana|   29520128|
|North Carolina|   31034602|
|          Utah|   30374790|
|       Montana|   28757725|
|      Nebraska|   28074526|
|      Illinois|   30059586|
|        Hawaii|   29991543|
|       Vermont|   28716560|
|      Oklahoma|   28843743|
|       Wyoming|   30308825|
|      Michigan|   29155589|
|      Virginia|   30220875|
|        Alaska|   28323539|
|  South Dakota|   30040657|
|  North Dakota|   30107726|
|     Louisiana|   30751912|
|   Mississippi|   29245707|
| Massachusetts|   30024449|
+--------------+-----------+
only showing top 20 rows
Time: 2.58s
== Parsed Logical Plan ==
'Aggregate ['state], ['state, 'sum('total_amount) AS total_sales#12132]
+- Project [customer_id#11784, product_id#11782, order_id#11783L, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790,

| Feature | Repartition (Round-Robin) | Repartition (By Column) | Coalesce |
|---------|----------------------------|--------------------------|----------|
| Full Shuffle | Yes | Yes | No  |
| Data Distribution | Evenly distributed across partitions | Distributed based on the specified column | Not guaranteed |
| Can Increase Partitions | Yes | Yes | No |
| Can Decrease Partitions | Yes | Yes | Yes |
| Physical Plan | `Exchange RoundRobinPartitioning` | `Exchange hashpartitioning(column)` | `Coalesce` |
| Primary Use Case | Evenly rebalance data | Prepare data for joins or aggregations on a key | Reduce the number of output partitions/files |


### Observation

#### Plan Comparison: ENSURE_REQUIREMENTS vs REPARTITION_BY_NUM

Both physical plans show `Exchange hashpartitioning(customer_id, 10)` on both sides of the join — but with a key difference in the tag:

- **Baseline:** `Exchange hashpartitioning(customer_id, 10), ENSURE_REQUIREMENTS`
- **Repartition-by-column:** `Exchange hashpartitioning(customer_id, 10), REPARTITION_BY_NUM`

`ENSURE_REQUIREMENTS` means Spark's optimizer introduced the shuffle automatically to satisfy the SortMergeJoin requirement.
`REPARTITION_BY_NUM` means our explicit `repartition(10, col("customer_id"))` call was executed.

The shuffle cost is identical in both cases — which is why execution times are nearly the same (3.16s vs 3.53s). The repartition version is marginally slower because the explicit repartition is treated as a separate plan step before join planning begins.

#### Why Didn't the Repartition Help Here?

For a single isolated join, Spark already shuffles both sides on the join key automatically as part of SortMergeJoin planning. Explicitly pre-partitioning by column does not eliminate that shuffle — it only changes who initiated it.

#### When Does Repartition by Column Actually Help?

The real benefit appears when the same pre-partitioned DataFrame is reused across multiple joins on the same key in a single pipeline:

```python
# Pay the shuffle cost once
orders_repart = orders_df.repartition(10, col("customer_id"))
orders_repart.cache()

# Both joins reuse the existing partitioning — no re-shuffle on orders_repart
result1 = orders_repart.join(customers_df, "customer_id")
result2 = orders_repart.join(returns_df, "customer_id")

# Without pre-partitioning, each join independently shuffles
# the orders DataFrame — paying the cost twice
```

In a multi-join pipeline, `repartition(n, col("key"))` combined with `cache()` means the shuffle is paid once and the partitioning is reused for every subsequent join on that key.

#### Key Takeaway

 The difference between `ENSURE_REQUIREMENTS` and `REPARTITION_BY_NUM` is subtle but meaningful: one tells you Spark chose the shuffle, the other tells you that you did.

### 7) Cache vs Persist
Scenario

The orders dataset is used multiple times for different aggregations.

Without caching, Spark reads the Parquet file and recomputes the entire lineage for every action.

By caching/persisting the DataFrame, Spark can reuse the computed data for subsequent actions, reducing execution time.

### 7.1 Baseline Implementation

The same DataFrame is used in multiple actions without caching. Spark recomputes the execution plan for each action by reading the source data again.

In [260]:
t0 = time.time()

orders_df.groupBy("order_year") \
    .agg(sum("total_amount").alias("sales")) \
    .show()

orders_df.groupBy("order_month") \
    .agg(avg("total_amount").alias("avg_sales")) \
    .show()

orders_df.groupBy("product_id") \
    .agg(count("*").alias("orders")) \
    .show()

print(f"Time: {time.time()-t0:.2f}s")

+----------+---------+
|order_year|    sales|
+----------+---------+
|      2025|499045433|
|      2026| 42307113|
|      2024|501163315|
|      2023|454713728|
+----------+---------+

+-----------+------------------+
|order_month|         avg_sales|
+-----------+------------------+
|          3|1501.0154887076242|
|         10|1492.8865844986963|
|          1|1490.8676302821457|
|          8| 1494.225660697056|
|         12|1501.9521205435726|
|          5|1495.0689500888957|
|          7|1501.3013309335481|
|          6|1504.3698969420918|
|         11|1497.2731998017575|
|          4|1495.0197712994132|
|          9|1501.3718604679698|
|          2| 1491.095440002081|
+-----------+------------------+

+----------+------+
|product_id|orders|
+----------+------+
|      2047|   167|
|      4788|   211|
|      4073|   188|
|       779|   196|
|      2739|   195|
|       306|   189|
|      2750|   212|
|      2123|   194|
|      3766|   208|
|       586|   192|
|       676|   210|
|     

### 7.2 Cache Implementation

The DataFrame is cached in memory after the first action. Subsequent actions reuse the cached data instead of re-reading the Parquet files.


In [261]:

orders_cache = orders_df.cache()

# Materialize the cache
orders_cache.count()

t0 = time.time()

orders_cache.groupBy("order_year") \
    .agg(sum("total_amount").alias("sales")) \
    .show()

orders_cache.groupBy("order_month") \
    .agg(avg("total_amount").alias("avg_sales")) \
    .show()
orders_cache.groupBy("product_id") \
    .agg(count("*").alias("orders")) \
    .show()

print(f"Time: {time.time()-t0:.2f}s")



+----------+---------+
|order_year|    sales|
+----------+---------+
|      2025|499045433|
|      2026| 42307113|
|      2024|501163315|
|      2023|454713728|
+----------+---------+

+-----------+------------------+
|order_month|         avg_sales|
+-----------+------------------+
|          3|1501.0154887076242|
|         10|1492.8865844986963|
|          1|1490.8676302821457|
|          8| 1494.225660697056|
|         12|1501.9521205435726|
|          5|1495.0689500888957|
|          7|1501.3013309335481|
|          6|1504.3698969420918|
|         11|1497.2731998017575|
|          4|1495.0197712994132|
|          9|1501.3718604679698|
|          2| 1491.095440002081|
+-----------+------------------+

+----------+------+
|product_id|orders|
+----------+------+
|      2047|   167|
|      4788|   211|
|      4073|   188|
|       779|   196|
|      2739|   195|
|       306|   189|
|      2750|   212|
|      2123|   194|
|      3766|   208|
|       586|   192|
|       676|   210|
|     

### 7.3 Persist Implementation

Unlike cache(), persist() allows choosing the storage level. In this example, MEMORY_AND_DISK is used, enabling Spark to spill partitions to disk if memory is insufficient.

In [262]:
from pyspark import StorageLevel

In [263]:
orders_persist = orders_df.persist(StorageLevel.MEMORY_AND_DISK)

# Materialize
orders_persist.count()

t0 = time.time()

orders_persist.groupBy("order_year") \
    .agg(sum("total_amount").alias("sales")) \
    .show()

orders_persist.groupBy("order_month") \
    .agg(avg("total_amount").alias("avg_sales")) \
    .show()

orders_persist.groupBy("product_id") \
    .agg(count("*").alias("orders")) \
    .show()

print(f"Time: {time.time()-t0:.2f}s")

orders_persist.unpersist()


+----------+---------+
|order_year|    sales|
+----------+---------+
|      2025|499045433|
|      2026| 42307113|
|      2024|501163315|
|      2023|454713728|
+----------+---------+

+-----------+------------------+
|order_month|         avg_sales|
+-----------+------------------+
|          3|1501.0154887076242|
|         10|1492.8865844986963|
|          1|1490.8676302821457|
|          8| 1494.225660697056|
|         12|1501.9521205435726|
|          5|1495.0689500888957|
|          7|1501.3013309335481|
|          6|1504.3698969420918|
|         11|1497.2731998017575|
|          4|1495.0197712994132|
|          9|1501.3718604679698|
|          2| 1491.095440002081|
+-----------+------------------+

+----------+------+
|product_id|orders|
+----------+------+
|      2047|   167|
|      4788|   211|
|      4073|   188|
|       779|   196|
|      2739|   195|
|       306|   189|
|      2750|   212|
|      2123|   194|
|      3766|   208|
|       586|   192|
|       676|   210|
|     

DataFrame[product_id: int, order_id: bigint, customer_id: int, quantity: int, order_date: date, price: int, total_amount: int, order_year: int, order_month: int]

### Observation

Without caching, Spark recomputed the DataFrame for every action, repeatedly scanning the Parquet source.

After applying `cache()`, Spark stored the DataFrame in memory after the first action, allowing subsequent actions to reuse the cached data.

Using `persist(StorageLevel.MEMORY_AND_DISK)` provided similar performance while allowing Spark to spill partitions to disk when memory was insufficient.

This demonstrates that caching and persisting are beneficial when the same DataFrame is reused multiple times within a Spark application.

| Feature | Cache | Persist |
|---------|-------|----------|
| Default Storage Level | MEMORY_AND_DISK (DataFrame API) | User-defined |
| Custom Storage Level | No | Yes |
| Can Spill to Disk | Yes (default DataFrame behavior) | Depends on storage level |
| Best Use Case | Frequently reused DataFrames | Large DataFrames or custom storage requirements |
| Release Memory | `unpersist()` | `unpersist()` |

### 8) Built-in Functions vs Python UDF

### 8.1 Baseline Implementation (Python UDF)

In [264]:
def sales_category(amount):
    if amount < 1000:
        return "Low"
    elif amount < 2000:
        return "Medium"
    else:
        return "High"

sales_udf = udf(sales_category, StringType())

t0 = time.time()

result = (
    orders_df
    .withColumn("category", sales_udf(col("total_amount")))
    .groupBy("category")
    .agg(count("*").alias("orders"))
)

result.show()

print(f"Time: {time.time()-t0:.2f}s")

result.explain(True)

+--------+------+
|category|orders|
+--------+------+
|    High|284426|
|     Low|458693|
|  Medium|256881|
+--------+------+

Time: 11.18s
== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'count(*) AS orders#14242]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, sales_category(total_amount#11788)#14240 AS category#14241]
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
category: string, orders: bigint
Aggregate [category#14241], [category#14241, count(1) AS orders#14242L]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, sales_category(total_amount#11788)#14240 AS category#14241]
   

8.2) Optimized Implementation (Built-in Function)

In [265]:


t0 = time.time()

result = (
    orders_df
    .withColumn(
        "category",
        when(col("total_amount") < 1000, "Low")
        .when(col("total_amount") < 2000, "Medium")
        .otherwise("High")
    )
    .groupBy("category")
    .agg(count("*").alias("orders"))
)

result.show()

print(f"Time: {time.time()-t0:.2f}s")

result.explain(True)

+--------+------+
|category|orders|
+--------+------+
|    High|284426|
|     Low|458693|
|  Medium|256881|
+--------+------+

Time: 1.96s
== Parsed Logical Plan ==
'Aggregate ['category], ['category, 'count(*) AS orders#14266]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790, CASE WHEN (total_amount#11788 < 1000) THEN Low WHEN (total_amount#11788 < 2000) THEN Medium ELSE High END AS category#14265]
   +- Relation [product_id#11782,order_id#11783L,customer_id#11784,quantity#11785,order_date#11786,price#11787,total_amount#11788,order_year#11789,order_month#11790] parquet

== Analyzed Logical Plan ==
category: string, orders: bigint
Aggregate [category#14265], [category#14265, count(1) AS orders#14266L]
+- Project [product_id#11782, order_id#11783L, customer_id#11784, quantity#11785, order_date#11786, price#11787, total_amount#11788, order_year#11789, order_month#11790,

| Implementation    | Expected Observation                                                                                      |
| ----------------- | --------------------------------------------------------------------------------------------------------- |
| Python UDF        | Physical plan contains `PythonUDF`; generally slower due to JVM-Python communication |
| Built-in Function | No Python execution in the plan; Catalyst optimizes the expression and execution is usually faster        |


### Observation

Python UDFs require Spark to serialize each row from the JVM into a Python
process, execute the function, then deserialize the result back row by row.
This serialization overhead is the bottleneck, not the function logic itself.

Built-in functions (`when`, `otherwise`) execute entirely inside the JVM using
Spark's Catalyst optimizer and whole-stage code generation. No serialization
happens. The physical plan confirms this the UDF appears as a black-box
`PythonUDF` node that Catalyst cannot optimize, while `when/otherwise` compiles
down to a native `CASE WHEN` expression inside a `Project` node.

**Rule:** Always prefer built-in functions over Python UDFs. If custom logic is
unavoidable, use a Pandas UDF (vectorized UDF) it operates on Arrow batches
instead of individual rows, making it significantly faster than a row-level UDF
while still allowing Python logic.

| | Python UDF | Pandas UDF | Built-in Function |
|---|---|---|---|
| Execution | Row by row | Arrow batch | JVM native |
| Catalyst optimization |  No |  No |  Yes |
| Serialization cost | High | Low | None |
| Use when | Avoid | Custom logic needed | Always prefer |

## Summary — PySpark Optimization Techniques

The following table consolidates all optimization techniques demonstrated in this notebook,
the problem each solves, and the signal to look for in the physical execution plan.

| # | Technique | Problem Solved | Key Signal in explain() |
|---|-----------|---------------|------------------------|
| 1 | Broadcast Join | Shuffle on both sides of a join | `BroadcastHashJoin` replaces `SortMergeJoin` |
| 2 | Predicate Pushdown | Full file scan before filtering | `PushedFilters: [...]` in FileScan |
| 3 | Column Pruning | Reading unused columns from Parquet | `ReadSchema` lists only required columns |
| 4 | Partition Pruning | Scanning all partitions unnecessarily | `PartitionFilters: [order_year = 2025]` |
| 5 | Data Skew + Salting | One partition processing all hot-key rows | Even key distribution after salting |
| 6 | Repartition vs Coalesce | Wrong partitioning strategy for the use case | `RoundRobinPartitioning` vs `Coalesce` node |
| 7 | Cache / Persist | Recomputing the same DataFrame multiple times | `is_cached = True` after materialisation |
| 7 | UDF vs Built-In Function | Extra time in JVM-Python communication | `PythonUDF` while using UDF|

---

## Key Principles

**1. Always verify with explain()**  
Never assume an optimization worked. Check the physical plan — look for
`BroadcastHashJoin`, `PartitionFilters`, `PushedFilters`, and `ReadSchema`
to confirm the optimizer applied what you intended.

**2. Reduce data as early as possible**  
Filter before joins. Select only required columns before transformations.
Every byte eliminated before a shuffle saves network transfer and memory.

**3. Shuffles are the most expensive operation**  
Every `Exchange` node in the physical plan is a shuffle — data moving across
the network between executors. Count the Exchange nodes before and after
an optimization. Fewer Exchange nodes = faster job.

**4. Understand what AQE does and does not handle**  
AQE (Adaptive Query Execution) handles join skew and shuffle partition
coalescing automatically in Spark 3.x. It does **not** handle groupBy skew —
that requires explicit salting. Never rely on AQE as a substitute for
understanding your data distribution.

**5. Single-node benchmarks have limits**  
Optimizations like salting, repartition-by-column, and broadcast joins show
their full benefit on multi-node clusters where network shuffle and executor
imbalance are real costs. On a single-node environment, the proof is in the
execution plan and key distribution — not wall-clock time.

